Importing Libraries

In [2]:
import requests
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

Loading playlist data

In [3]:
playlists = pd.read_csv("playlists.csv")
playlists.head()

,track_id,track_name,playlist_id,playlist_name
0,4qEoqyPbLYnLOii6mKlIjI,"Determinate - From ""Lemonade Mouth""",7jQHOrErpLMStcUUSavQWR,Post teen pop
1,5lz0NiPw32Gq4kMIUJvuw2,"Take On the World - Theme Song From ""Girl Meet...",7jQHOrErpLMStcUUSavQWR,Post teen pop
2,1rM0CnyUiiw6A9CHJRXjZA,Chillin' Like a Villain,7jQHOrErpLMStcUUSavQWR,Post teen pop
3,744ZuzjXQmoJmOdk2I1ym9,"Keep It Undercover - Theme Song From ""K.C. Und...",7jQHOrErpLMStcUUSavQWR,Post teen pop
4,6VcKAd94eVxNyudlWN91GH,"Time of Our Lives - Main Title Theme From ""I D...",7jQHOrErpLMStcUUSavQWR,Post teen pop


Data Preparation for evaluation : Grouping the data by Playlist ID and Name

In [4]:
# Grouping by the playlist id and name
# converting track ids into a list for each playlist
playlist_data = playlists.groupby(['playlist_id', 'playlist_name'])['track_id'].apply(list).reset_index()
playlist_data.head()

,playlist_id,playlist_name,track_id
0,0BwUQpqHSlC2YfKwOp2dQV,Gangsta Rap ð Rap Party,"[5jAeM4WKpqDX9wDJGa7IFq, 3QbglwFgPJDHq7TbmUb3I..."
1,0DdGqNa18DwYyfIR05OrW1,Viral Southern Hip Hop,"[53v6w87pm4hVFwC7GlT2JI, 0M9mH33xccGM4ITZIGwX2..."
2,0QN4FeJQ1mpCygRg9r2JIK,Permanent wave ð,"[6zfczP87XO2SxWlQtnjFNa, 6iRrt1UtrgsxsOZUjp6Vg..."
3,0TT57Pe3RZNGCy98G1UQpM,TROPICALð´,"[3zbSYBDUbDEbqUIlCmOw09, 1siV5UzcXuFK5tMUVQ5Ki..."
4,0VS9w0NY4KXfLORkhY81s8,Permanent Wave Cafe,"[42T2QQv3xgBlpQxaSP7lnK, 3jxrPXWriZPIXKGbuniXh..."


The number of playlists to be used in evaluation.

In [5]:
print ("Number of playlists to be used in evaluation : ",len(playlist_data))

Number of playlists to be used in evaluation :  51


Getting the tracks of each playlist ready for the evaluation

In [6]:
input_tracks_list = [] # a variable to store the lists of input_tracks from each playlist
expected_tracks_list = [] # a variable to store the lists of expected_tracks (the ground truth) from each playlist

for _, row in playlist_data.iterrows():
    # get tracks of each playlist
    tracks = row['track_id'].copy()

    # split the tracks of each playlist (80%, 20%)
    # 80% for input
    # 20% for expected tracks ground truth
    # reproduce the same split data everytime
    input_tracks, expected_tracks = train_test_split(tracks, test_size=0.2, random_state=42)
    
    # storing the lists of tracks from each playlist in the list
    input_tracks_list.append(input_tracks)
    expected_tracks_list.append(expected_tracks)
    
    

In [7]:
print (input_tracks_list[0])
print ("Number of the lists of input_tracks : ", len(input_tracks_list))
print ("Number of the lists of expected_tracks : ", len(expected_tracks_list))

['6djzhYE1NxgS7YgnZZEqnN', '3N5XOukuZ6JCJHoHVRowS7', '5HEm9RTM8fIuzKa1RBj6xZ', '5jAeM4WKpqDX9wDJGa7IFq', '7JXLHs3JxQ0sibWrYeV1co', '1RqpijoxxQJ9FWS1V56DeE', '6AFWx8JqsslV2MsWhpGjnR', '5sH7vJ3shCpWGpsHMjwkfN', '3BYY2756aaf3cCfrqaVRAt', '4D2pxxPpgyJdowXmebsXKs', '3QbglwFgPJDHq7TbmUb3IE', '2jv5jGgC1iFQ9Ury2ci9Y4', '7ucZwIC4XrQZf8L1yqCOGD', '33OjMQub4EtjwxY4Pi0g3U', '0zLXP2PI5JG3pZ2KaqEWS6', '4TnjEaWOeW0eKTKIEvJyCa', '5LUNEABcjkQ7jCSpu3uBPq', '0Lfz9oMafqp2UFH8xkZgDY', '3YreXzJAZRrchO72v5G5K9', '1VAFud1nSkqa7efJBxONAz', '3OueBUZsmQERpu4hSF0jCr', '0uYmQ3X53P03KWj83u5I59', '3cyJAZm2r79wdzrsQtv5Ob', '1fG01hFiFGXZKRMHXsBlyf']
Number of the lists of input_tracks :  51
Number of the lists of expected_tracks :  51


In [8]:
# find TP for Precision and Recall and calculate their scores
#  recommended (a list of track ids recommended by the model)
#  expected (a list of track ids from remaining 20% of a playlist)
#  k (the number of recommendation generated by the model)
#  numerator (the relevancy)
#  the denominator (TP + FP <or> TP + FN)
def find_score (recommended, expected, k, numerator, denominator):
    # for every track_id in the recommended list, if that id also exists in expected, the id is relevant id.
    for i in range(min(k, len(recommended))):
        if recommended[i] in expected:
            numerator += 1       # + 1 relevancy
    
    r = numerator/denominator       # formula
    return r

# PRECISION
#  recommended (a list of track ids recommended by the model)
#  expected (a list of track ids from remaining 20% of a playlist)
#  k (the number of recommendation generated by the model)
def precision_at_k(recommended, expected, k):
    # Precision@k = TP/(TP + FP) <How many of the recommendations are relevant>
    # TP + FP = k 
    denominator = min(k, len(recommended))      # handling with min just in case. (length of recommended is surely 10)
    numerator = 0 

    return find_score(recommended, expected, k, numerator, denominator)

# RECALL
#  recommended (a list of track ids recommended by the model)
#  expected (a list of track ids from remaining 20% of a playlist)
#  k (the number of recommendation generated by the model)
def recall_at_k (recommended, expected, k):
    # recall@k = TP/(TP + FN) <How many of the relevant tracks are predicted>
    # TP + FN = Number of tracks in the expected list
    denominator = len(expected)
    numerator = 0

    if denominator == 0:        #edge case
        return 0.0
    
    return find_score(recommended, expected, k, numerator, denominator)

# NDCG@k 
    # NDCG@K = DCG@K/IDCG
    # DCG@K = Sum of (relevance<1 or 0>)/log(base 2)(i + 1)     for every ith recommendation
    # IDCG = Sum of (relevance< always 1>)/log(base 2)(i + 1)     # this represents perfect recommendations

    # relevance : is the track relevant?
    # log(base 2) (i + 1) : is the relevant track's ranking good? 
    # [(smaller i >> smaller log value) >> larger (relevance/log(base 2)(i + 1)) result] 

# DCG@K
#  recommended (a list of track ids recommended by the model)
#  expected (a list of track ids from remaining 20% of a playlist)
#  k (the number of recommendation generated by the model)
def dcg_at_k (recommended, expected, k):
    dcg = 0.0
    for i in range(min(k, len(recommended))):
        if recommended[i] in expected:      # relevant condition
            dcg += 1 /math.log2(i + 1 + 1)      # adding (index starts from 0, so use (i+1))
            # if irrelevant, the numerator is 0, so, no need to add to dcg.
    return dcg      # final dcg value after all loop

# IDCG@k
#  recommended (a list of track ids recommended by the model)
#  expected (a list of track ids from remaining 20% of a playlist)
#  k (the number of recommendation generated by the model)
def idcg_at_k (expected, k):
    idcg = 0.0
    # if k is 10 and expected (20% playlist) is smaller than that, the best result should be hitting 3 relevant tracks, not 10. so, min between them is used.
    max_hit = min(len(expected), k)     
    for i in range(max_hit):      # this is best case, so all are set as relevant, 
        idcg += 1 /math.log2(i + 1 + 1)     #so add every time (index starts from 0, so use (i+1))
    return idcg     # final idcg value after all loop

# NDCG@K
# dcg (DCG@K)
# idcg (IDCG@K)
def ndcg_at_k(recommeded, expected, k):
    dcg = dcg_at_k(recommeded, expected, k)
    idcg = idcg_at_k(expected, k)

    if idcg == 0.0:
        return 0.0
    return dcg/idcg    # final NDCG@K 
  

# find Metrics Scores for a model
def find_all_scores(recommended, expected, k):
    p = precision_at_k(recommended, expected, k)
    r = recall_at_k(recommended, expected, k)
    ncdg = ndcg_at_k(recommended, expected, k)
    return p, r, ncdg

Define the function which will evaluate the model. 
Inputs are 
- the URL of the recommender model, 
- input_tracks_list which includes the lists of ids, [[],[],[],...]
- top-k recommendations 

Outputs are 
- the precision list which includes the values of precision for each playlist.
- the recall list which includes the values of recall for each playlist.
- the ncdg list which includes the values of ncdg for each playlist.

This function will get called for each model to get the lists of results which the average will then be calculated.


In [27]:
def evaluate_model(api_url, input_tracks_list, expected_tracks_list, top_k):

    all_precisions = []
    all_recalls = []
    all_ncdg = []

    params_value = {"k": top_k}

    for index, tracks_list in enumerate(input_tracks_list): 

        # no pref
        json_payload_no_pref = {
            "track_ids" : tracks_list
        }

        # data request & get list of recommended track ids
        response = requests.post(api_url, params=params_value, json= json_payload_no_pref)     # post with input to url
        data = response.json()              # get the current playlist's output/recommended tracks (JSON data)
        recommended_ids = [d["track_id"] for d in data]     # get list of recommended ids
        
        # expected track ids (continuation list)
        expected_ids = expected_tracks_list[index]      # expected tracks ids of current playlist

        # compare the recommended_ids with the other 20% (expected_ids) of the playlist
        precision, recall, ncdg = find_all_scores(recommended_ids, expected_ids, top_k)   

        # adding current playlist's scores to the list of all scores (for later average calculation)
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_ncdg.append(ncdg)

    return all_precisions, all_recalls, all_ncdg




    
    

Find the precision, recall, NCDG results for each model.

In [33]:
top_k = 15 # top k value
# Euclidean API URL
modelEuclideanURL = "http://127.0.0.1:8080/api/recommend/"
baselineCosineURL = "http://127.0.0.1:8080/api/recommend/baseline/cosine/"
baselineRandomByArtistURL = "http://127.0.0.1:8080/api/recommend/baseline/random_by_artist/"
baselineRandomURL = "http://127.0.0.1:8080/api/recommend/baseline/random/"


precision_E, recall_E, ncdg_E = evaluate_model(modelEuclideanURL, input_tracks_list, expected_tracks_list, top_k)
precision_C, recall_C, ncdg_C = evaluate_model(baselineCosineURL, input_tracks_list, expected_tracks_list, top_k)
precision_RA, recall_RA, ncdg_RA = evaluate_model(baselineRandomByArtistURL, input_tracks_list, expected_tracks_list, top_k)
precision_R, recall_R, ncdg_R = evaluate_model(baselineRandomURL, input_tracks_list, expected_tracks_list, top_k)

Finding the average(Overall) scores for the models

In [34]:
def find_avg_scores(score_list):
    if len(score_list) == 0:
        return 0
    return sum(score_list)/len(score_list)

def find_avg_for_all_models(p_score, r_score, ncdg_score):
    avg_p = find_avg_scores(p_score)
    avg_r = find_avg_scores(r_score)
    avg_ncdg = find_avg_scores(ncdg_score)
    return avg_p, avg_r, avg_ncdg


# Euclidean model's average Scores 
avg_precision_E, avg_recall_E, avg_ncdg_E = find_avg_for_all_models(precision_E, recall_E, ncdg_E)

# Cosine model's average Scores 
avg_precision_C, avg_recall_C, avg_ncdg_C = find_avg_for_all_models(precision_C, recall_C, ncdg_C)

# Random by artist model's average Scores 
avg_precision_RA, avg_recall_RA, avg_ncdg_RA = find_avg_for_all_models(precision_RA, recall_RA, ncdg_RA)

# Random model's average Scores 
avg_precision_R, avg_recall_R, avg_ncdg_R = find_avg_for_all_models(precision_R, recall_R, ncdg_R)

print ("Overall PRECISION Score of EUCLIDEAN : ", avg_precision_E)
print ("Overall RECALL Score of EUCLIDEAN : ", avg_recall_E)
print ("Overall NCDG Score of EUCLIDEAN : ", avg_ncdg_E)
print ("Overall PRECISION Score of COSINE : ", avg_precision_C)
print ("Overall RECALL Score of COSINE : ", avg_recall_C)
print ("Overall NCDG Score of COSINE : ", avg_ncdg_C)
print ("Overall PRECISION Score of RANDOM BY ARTIST : ", avg_precision_RA)
print ("Overall RECALL Score of RANDOM BY ARTIST : ", avg_recall_RA)
print ("Overall NCDG Score of RANDOM BY ARTIST : ", avg_ncdg_RA)
print ("Overall PRECISION Score of RANDOM : ", avg_precision_R)
print ("Overall RECALL Score of RANDOM : ", avg_recall_R)
print ("Overall NCDG Score of RANDOM : ", avg_ncdg_R)


Overall PRECISION Score of EUCLIDEAN :  0.03529411764705882
Overall RECALL Score of EUCLIDEAN :  0.10583566760037351
Overall NCDG Score of EUCLIDEAN :  0.08166461174494949
Overall PRECISION Score of COSINE :  0.03006535947712418
Overall RECALL Score of COSINE :  0.08900560224089636
Overall NCDG Score of COSINE :  0.070631115104561
Overall PRECISION Score of RANDOM BY ARTIST :  0.15032679738562094
Overall RECALL Score of RANDOM BY ARTIST :  0.3892857142857143
Overall NCDG Score of RANDOM BY ARTIST :  0.40261056337453793
Overall PRECISION Score of RANDOM :  0.00130718954248366
Overall RECALL Score of RANDOM :  0.004901960784313725
Overall NCDG Score of RANDOM :  0.0019136276960397141


Make table for comparison

In [ ]:
results = {
    "Model": ["Euclidean", "Cosine", "Random by Artist", "Random"],
    "Precision": [avg_precision_E, avg_precision_C, avg_precision_RA, avg_precision_R],
    "Recall": [avg_recall_E, avg_recall_C, avg_recall_RA, avg_recall_R],
    "NDCG": [avg_ncdg_E, avg_ncdg_C, avg_ncdg_RA, avg_ncdg_R]
}

results_comparison_df = pd.DataFrame(results)

results_comparison_df.to_csv("topK15_evaluation_results_before_recommender_logic_update (First_recommender_logic_results).csv", index=False)

results_comparison_df

,Model,Precision,Recall,NDCG
0,Euclidean,0.035294,0.105836,0.081665
1,Cosine,0.030065,0.089006,0.070631
2,Random by Artist,0.150327,0.389286,0.402611
3,Random,0.001307,0.004902,0.001914
